In [1]:
from loqs.backends import QSimQuantumState, PyGSTiNoiseModel
from loqs.core import QuantumProgram

from loqs.codepacks import code_5_1_3_quantinuum2022 as code_5_1_3

import pygsti

In [2]:
# In theory, codepacks could take arguments here to generate a family of codes
# E.g. Auxiliary qubit reuse (surface 13 vs 17), syndrome extraction schedules, etc
code_5q = code_5_1_3.create_qec_code()

In [3]:
list(code_5q.instructions.keys())

['Non-FT Minus Prep',
 'FT Minus Prep',
 'X',
 'Z',
 'H',
 'Logical Prime Basis Transform',
 'Logical Prime Basis Inverse Transform',
 'Non-FT Logical Z Measure',
 'Non-FT Logical X Measure',
 'Non-FT Minus Unprep',
 'Adaptive Measure Termination',
 'Adaptive Measure Part III',
 'Adaptive Measure Part II',
 'Adaptive Measure Part I',
 'Adaptive Measure']

In [4]:
# Define a pyGSTi processor spec and noise model
# This is the only pyGSTi required here,
# and could eventually be traded out for something else
qubits = ["A0", "A1"] + [f"D{i+2}" for i in range(5)]
gate_names = ["Gxpi", "Gypi", "Gzpi", "Gzpi2", "Gzmpi2", "Gh", "Gcnot", "Gcphase"]

# TODO: Currently Iz does not need to be set here
# This is because QSimQuantumState actually does not try to pull the rep
# Otherwise, this would result in a KeyError in PyGSTiNoiseModel.get_reps(),
# since it technically should be provided
pspec = pygsti.processors.QubitProcessorSpec(
    len(qubits), 
    gate_names=gate_names,
    qubit_labels=qubits,
    availability={k: "all-permutations" for k in gate_names}
)

ideal_model_pygsti = pygsti.models.create_crosstalk_free_model(pspec)

ideal_model = PyGSTiNoiseModel(ideal_model_pygsti)


In [5]:
# Logical quantum program information
patch_types = {"5Q": code_5q}

# Stack
# The "Init *" instructions will be generated by the QuantumProgram based on the values
# of state_type and patch_types below
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_type
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("Non-FT Logical X Measure", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=ideal_model,
    state_type=QSimQuantumState,
    patch_types=patch_types,
    name="Prep minus, measure X"
)

In [6]:
# We can do a "dry run" which skips actually propagating the state forward
# This ensures that all instructions set the correct thing in the history for future instructions, etc.
program.run(dry_run=True)

Dry run completed successfully!


In [7]:
# And now we can run it for real!
program.run(shots=100)

Program Prep minus, measure X: 100%|██████████| 100/100 [00:00<00:00, 140.74it/s]


In [8]:
from collections import Counter

# collect_shot_data lets us pull out keys from the histories of every shot
# In this case, we pull logical_measurement from the last frame (-1) of each shot
# We pass it into Counter to get more convenient 0/1 counts
Counter(program.collect_shot_data("logical_measurement", -1))

# Because we prepped logical minus and did a logical X measurement, we expect to get all 1s here

Counter({1: 100})

In [9]:
# Make a version that goes to logical 0 and measures in Z
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_type
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("H", "L0"),
    ("X", "L0"),
    ("Non-FT Logical Z Measure", "L0")
]

# We want to use the same program settings, just swap out the stack and name
program2 = QuantumProgram.from_quantum_program(program, stack, name="Prep 0, measure Z")

program2.run(shots=100)

Counter(program2.collect_shot_data("logical_measurement", -1))

Program Prep 0, measure Z: 100%|██████████| 100/100 [00:00<00:00, 155.74it/s]


Counter({0: 100})

In [10]:
# Now let's stay in minus but measure in Z
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_type
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("Non-FT Logical Z Measure", "L0")
]

# This one will be non-determinate outcome. Let's seed the RNG
program3 = QuantumProgram.from_quantum_program(program, stack, default_base_seed=20240702, name="Prep -, measure Z")

program3.run(shots=1000) # Let's also collect more statistics!

Counter(program3.collect_shot_data("logical_measurement", -1))

Program Prep -, measure Z: 100%|██████████| 1000/1000 [00:05<00:00, 197.61it/s]


Counter({0: 517, 1: 483})

In [11]:
# And now we can show that it's deterministic counts with the same RNG seed
program4 = QuantumProgram.from_quantum_program(program3, name="Prep -, measure Z... again")

program4.run(shots=1000) # Let's also collect more statistics!

Counter(program4.collect_shot_data("logical_measurement", -1))

Program Prep -, measure Z... again: 100%|██████████| 1000/1000 [00:05<00:00, 199.44it/s]


Counter({0: 517, 1: 483})